In [ ]:
from IPython.display import HTML

HTML("""
<div id="container" style="background: #111; padding: 20px; color: #0f0; font-family: monospace; border-radius: 10px; text-align: center;">
    <div id="log">SISTEMA: Esperando interacción...</div>
    <button id="startBtn" style="padding: 10px 20px; margin: 10px; cursor: pointer; background: #0f0; border: none; font-weight: bold;">INICIAR CÁMARA</button>
    <br>
    <video id="v" width="640" height="480" autoplay playsinline style="display:none;"></video>
    <canvas id="c" width="640" height="480" style="border: 2px solid #0f0; max-width: 100%; transform: scaleX(-1);"></canvas>
</div>

<script src="https://cdn.jsdelivr.net/npm/@mediapipe/face_mesh/face_mesh.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/drawing_utils/drawing_utils.js"></script>

<script>
    const v = document.getElementById('v');
    const c = document.getElementById('c');
    const ctx = c.getContext('2d');
    const log = document.getElementById('log');
    const btn = document.getElementById('startBtn');

    async function startApp() {
        btn.style.display = 'none';
        log.innerText = "SISTEMA: Cargando modelos...";

        try {
            const fm = new FaceMesh({locateFile: (file) => `https://cdn.jsdelivr.net/npm/@mediapipe/face_mesh/${file}`});
            fm.setOptions({ maxNumFaces: 1, refineLandmarks: false, minDetectionConfidence: 0.5 });

            fm.onResults((res) => {
                log.innerText = "SISTEMA: Online";
                ctx.clearRect(0, 0, c.width, c.height);
                ctx.drawImage(res.image, 0, 0, c.width, c.height);
                if (res.multiFaceLandmarks) {
                    for (const p of res.multiFaceLandmarks) {
                        drawConnectors(ctx, p, FACEMESH_TESSELATION, {color: '#00FF0033', lineWidth: 1});
                    }
                }
            });

            const cam = new Camera(v, {
                onFrame: async () => { await fm.send({image: v}); },
                width: 640, height: 480
            });

            await cam.start();
            log.innerText = "SISTEMA: Cámara activa";
        } catch (e) {
            log.innerText = "ERROR: " + e.message;
            console.error(e);
        }
    }

    btn.onclick = startApp;
</script>
""")